In [1]:
# Cell 1: 패키지 설치 및 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')
!pip install -q lightgbm xgboost catboost

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.0 MB/s eta 0:00:00


In [7]:
# Cell 2: import 및 경로 설정, 유틸 함수(reduce_mem_usage)
import pandas as pd
import numpy as np
import lightgbm as lgb
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
import os
import sys
import warnings
import gc

# Force UTF-8 encoding for stdout
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

warnings.filterwarnings('ignore')

# Set paths (Colab Drive)
DATA_DIR = '/content/drive/MyDrive/MyDrive/LightGBM'
TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH = os.path.join(DATA_DIR, 'test.csv')
LAYOUT_PATH = os.path.join(DATA_DIR, 'layout_info.csv')
OUTPUT_PATH = os.path.join(DATA_DIR, 'submission_67.csv')

def reduce_mem_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max: df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max: df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max: df[col] = df[col].astype(np.int32)
                else: df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max: df[col] = df[col].astype(np.float32)
                else: df[col] = df[col].astype(np.float64)
    return df

In [8]:
# Cell 3: preprocess_data 함수 (Exp 66 원본 그대로, 수정 금지)
def preprocess_data(train, test, layout):
    print("Preprocessing data (Exp 67: Target Transform Comparison)...")
    train = train.merge(layout, on='layout_id', how='left')
    test = test.merge(layout, on='layout_id', how='left')

    le = LabelEncoder()
    all_layout_types = pd.concat([train['layout_type'], test['layout_type']])
    le.fit(all_layout_types.astype(str))
    train['layout_type'] = le.transform(train['layout_type'].astype(str))
    test['layout_type'] = le.transform(test['layout_type'].astype(str))

    for df in [train, test]:
        df['robot_utilization'] = df['robot_active'] / (df['robot_total'] + 1e-6)
        df['charger_utilization'] = df['robot_charging'] / (df['charger_count'] + 1e-6)
        df['aisle_pressure'] = df['congestion_score'] / (df['aisle_width_avg'] + 1e-6)
        df['intersection_density'] = df['intersection_count'] / (df['floor_area_sqm'] + 1e-6)
        df['pack_station_pressure'] = df['order_inflow_15m'] / (df['pack_station_count'] + 1e-6)
        df['bottleneck_risk'] = df['congestion_score'] * df['intersection_density'] / (df['aisle_width_avg'] + 1e-6)
        df['task_intensity'] = df['order_inflow_15m'] / (df['robot_active'] + 1e-6)

    layout_num_cols = ['aisle_width_avg', 'intersection_count', 'robot_total']
    key_ops = ['congestion_score', 'robot_active', 'order_inflow_15m']
    for l_col in layout_num_cols:
        for o_col in key_ops:
            train[f'{l_col}_x_{o_col}'] = train[l_col] * train[o_col]
            test[f'{l_col}_x_{o_col}'] = test[l_col] * test[o_col]

    momentum_cols = ['congestion_score', 'low_battery_ratio', 'robot_active']
    for col in momentum_cols:
        train[f'{col}_vel'] = train.groupby('scenario_id')[col].diff(1)
        test[f'{col}_vel'] = test.groupby('scenario_id')[col].diff(1)

    train['time_idx'] = train.groupby('scenario_id').cumcount()
    test['time_idx'] = test.groupby('scenario_id').cumcount()
    train = train.sort_values(['scenario_id', 'time_idx'])
    test = test.sort_values(['scenario_id', 'time_idx'])

    target_cols = [
        'order_inflow_15m', 'unique_sku_15m', 'robot_active', 'robot_idle',
        'robot_charging', 'battery_mean', 'battery_std', 'low_battery_ratio',
        'charge_queue_length', 'avg_charge_wait', 'congestion_score',
        'max_zone_density', 'blocked_path_15m', 'near_collision_15m',
        'fault_count_15m', 'avg_recovery_time', 'task_reassign_15m',
        'replenishment_overlap', 'pack_utilization', 'loading_dock_util',
        'staging_area_util', 'label_print_queue'
    ]
    for col in target_cols:
        first_val_tr = train.groupby('scenario_id')[col].transform('first')
        train[f'{col}_vs_start'] = train[col] / (first_val_tr + 1e-6)
        train[f'{col}_delta_start'] = train[col] - first_val_tr

        first_val_ts = test.groupby('scenario_id')[col].transform('first')
        test[f'{col}_vs_start'] = test[col] / (first_val_ts + 1e-6)
        test[f'{col}_delta_start'] = test[col] - first_val_ts

    for col in target_cols:
        if col not in train.columns:
            continue
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            cum_max = prev.groupby(df['scenario_id']).cummax()
            cum_min = prev.groupby(df['scenario_id']).cummin()
            df[f'{col}_vs_cummax'] = df[col] / (cum_max + 1e-6)
            df[f'{col}_vs_cummin'] = df[col] / (cum_min.abs() + 1e-6)

    for col in target_cols:
        if col not in train.columns:
            continue
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            cum_max = prev.groupby(df['scenario_id']).cummax()
            cum_min = prev.groupby(df['scenario_id']).cummin()
            cum_range = cum_max - cum_min
            df[f'{col}_position_in_range'] = ((df[col] - cum_min) / (cum_range + 1e-3)).clip(0, 2)

    # [EXP 48 FEATURES MAINTAINED]
    target_lag_cols = [
        'congestion_score', 'fault_count_15m', 'charge_queue_length',
        'low_battery_ratio', 'blocked_path_15m', 'avg_recovery_time',
        'near_collision_15m', 'pack_utilization'
    ]
    for col in target_lag_cols:
        if col not in train.columns:
            continue
        for df in [train, test]:
            for lag in [4, 5, 6, 7]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp = prev.groupby(df['scenario_id'])
            df[f'{col}_roll7_mean'] = grp.rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
            df[f'{col}_roll7_std'] = grp.rolling(7, min_periods=1).std().reset_index(level=0, drop=True)
            df[f'{col}_roll10_mean'] = grp.rolling(10, min_periods=1).mean().reset_index(level=0, drop=True)
            df[f'{col}_roll10_std'] = grp.rolling(10, min_periods=1).std().reset_index(level=0, drop=True)

    SEQ_COLS_BASE = ["order_inflow_15m", "unique_sku_15m", "robot_active", "low_battery_ratio", "charge_queue_length", "congestion_score", "fault_count_15m"]

    # [EXP 49 FEATURES MAINTAINED]
    remaining_cols = [c for c in SEQ_COLS_BASE if c not in target_lag_cols]
    for col in remaining_cols:
        for df in [train, test]:
            for lag in [4, 5]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)
            prev = df.groupby('scenario_id')[col].shift(1)
            grp = prev.groupby(df['scenario_id'])
            df[f'{col}_roll7_mean'] = grp.rolling(7, min_periods=1).mean().reset_index(level=0, drop=True)
            df[f'{col}_roll7_std']  = grp.rolling(7, min_periods=1).std().reset_index(level=0, drop=True)

    # [EXP 50 FEATURES MAINTAINED]
    for col in target_cols:
        for df in [train, test]:
            for lag in [8, 10]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)

    for col in target_lag_cols:
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            grp = prev.groupby(df['scenario_id'])
            df[f'{col}_roll14_mean'] = grp.rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
            df[f'{col}_roll14_std']  = grp.rolling(14, min_periods=1).std().reset_index(level=0, drop=True)

    # [EXP 51 FEATURES MAINTAINED]
    SEQ_COLS = target_cols
    for col in SEQ_COLS:
        for df in [train, test]:
            for lag in [12, 15]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)

    for col in target_lag_cols:
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            grp = prev.groupby(df['scenario_id'])
            df[f'{col}_roll20_mean'] = grp.rolling(20, min_periods=1).mean().reset_index(level=0, drop=True)
            df[f'{col}_roll20_std']  = grp.rolling(20, min_periods=1).std().reset_index(level=0, drop=True)

    # [EXP 52 FEATURES MAINTAINED]
    for col in SEQ_COLS:
        for df in [train, test]:
            for lag in [20, 24]:
                df[f'{col}_lag{lag}'] = df.groupby('scenario_id')[col].shift(lag)

    remaining_cols_exp52 = [c for c in SEQ_COLS if c not in target_lag_cols]
    for col in remaining_cols_exp52:
        for df in [train, test]:
            prev = df.groupby('scenario_id')[col].shift(1)
            grp = prev.groupby(df['scenario_id'])
            df[f'{col}_roll14_mean'] = grp.rolling(14, min_periods=1).mean().reset_index(level=0, drop=True)
            df[f'{col}_roll14_std']  = grp.rolling(14, min_periods=1).std().reset_index(level=0, drop=True)
            df[f'{col}_roll20_mean'] = grp.rolling(20, min_periods=1).mean().reset_index(level=0, drop=True)
            df[f'{col}_roll20_std']  = grp.rolling(20, min_periods=1).std().reset_index(level=0, drop=True)

    train_new_scen = train['scenario_id'].values != np.roll(train['scenario_id'].values, 1); train_new_scen[0] = True
    test_new_scen = test['scenario_id'].values != np.roll(test['scenario_id'].values, 1); test_new_scen[0] = True

    for col in SEQ_COLS_BASE:
        for lag in [1, 2]:
            tr_lag = train[col].shift(lag).values.copy(); ts_lag = test[col].shift(lag).values.copy()
            for l in range(lag):
                tr_lag[np.roll(train_new_scen, l)] = np.nan; ts_lag[np.roll(test_new_scen, l)] = np.nan
            train[f'{col}_lag{lag}'] = tr_lag; test[f'{col}_lag{lag}'] = ts_lag

        train[f'{col}_exp_mean'] = train.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())
        test[f'{col}_exp_mean'] = test.groupby('scenario_id')[col].transform(lambda x: x.shift(1).expanding().mean())

    train['time_ratio'] = train.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))
    test['time_ratio'] = test.groupby('scenario_id')['time_idx'].transform(lambda x: x / (x.max() + 1e-6))

    for df in [train, test]:
        df['congestion_ratio'] = df['congestion_score'] / (df['congestion_score_exp_mean'] + 1e-6)
        df['steps_remaining'] = df.groupby('scenario_id')['time_idx'].transform('max') - df['time_idx']

    train.fillna(0, inplace=True); test.fillna(0, inplace=True)
    return reduce_mem_usage(train), reduce_mem_usage(test)

In [9]:
# Cell 4: apply_smoothed_te 함수 (Exp 66 원본 그대로)
def apply_smoothed_te(df_tr, df_val, target_col, k=30):
    global_mean = df_tr[target_col].mean()
    agg = df_tr.groupby('layout_id')[target_col].agg(['mean', 'std', 'median', 'count']).reset_index()
    agg['layout_mean'] = (agg['count'] * agg['mean'] + k * global_mean) / (agg['count'] + k)
    agg.rename(columns={'std': 'layout_std', 'median': 'layout_median', 'count': 'layout_count'}, inplace=True)
    df_val = df_val.merge(agg[['layout_id', 'layout_mean', 'layout_std', 'layout_median', 'layout_count']], on='layout_id', how='left')
    df_tr = df_tr.merge(agg[['layout_id', 'layout_mean', 'layout_std', 'layout_median', 'layout_count']], on='layout_id', how='left')
    df_val['layout_mean'] = df_val['layout_mean'].fillna(global_mean)
    df_val['layout_std'] = df_val['layout_std'].fillna(df_tr[target_col].std())
    df_val['layout_median'] = df_val['layout_median'].fillna(df_tr[target_col].median())
    df_val['layout_count'] = df_val['layout_count'].fillna(0)
    return df_tr, df_val, ['layout_mean', 'layout_std', 'layout_median', 'layout_count']

In [ ]:
# Cell 5: 데이터 로드 및 전처리 실행
print("--- Experiment 67: Target Transform Comparison ---")
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
layout = pd.read_csv(LAYOUT_PATH)
common_layouts = set(train_raw['layout_id'].unique()) & set(test_raw['layout_id'].unique())

train, test = preprocess_data(train_raw, test_raw, layout)
TARGET = 'avg_delay_minutes_next_30m'
features_base = [c for c in train.columns if c not in ['ID', 'layout_id', 'scenario_id', TARGET, 'is_seen']]
train['is_seen'] = train['layout_id'].isin(common_layouts)

--- Experiment 67: Target Transform Comparison ---
Preprocessing data (Exp 67: Target Transform Comparison)...


In [ ]:
# Cell 6: zero_imp_features 및 features_pruned (Exp 66 원본 그대로)
zero_imp_features = [
    'charge_queue_length', 'avg_charge_wait', 'charge_queue_length_lag2', 'fault_count_15m_lag2',
    'time_ratio', 'task_reassign_15m_vs_cummin', 'low_battery_ratio_vel', 'low_battery_ratio_lag2',
    'task_reassign_15m', 'blocked_path_15m_lag8', 'blocked_path_15m_lag10', 'near_collision_15m_lag8',
    'near_collision_15m_lag10', 'fault_count_15m_lag8', 'fault_count_15m_lag10', 'avg_recovery_time_lag8',
    'avg_recovery_time_lag10', 'task_reassign_15m_lag8', 'task_reassign_15m_lag10', 'replenishment_overlap_lag8',
    'replenishment_overlap_lag10', 'robot_charging_lag10', 'low_battery_ratio_lag8', 'low_battery_ratio_lag10',
    'charge_queue_length_lag8', 'charge_queue_length_lag10', 'avg_charge_wait_lag8', 'avg_charge_wait_lag10',
    'fault_count_15m_vs_cummax', 'fault_count_15m_vs_cummin', 'avg_recovery_time_vs_cummax', 'task_reassign_15m_vs_cummax',
    'low_battery_ratio_vs_cummax', 'low_battery_ratio_vs_cummin', 'charge_queue_length_vs_cummin', 'avg_charge_wait_vs_cummax',
    'blocked_path_15m_vs_cummax', 'near_collision_15m_vs_cummin', 'charge_queue_length_roll14_mean', 'task_reassign_15m_vs_start',
    'avg_recovery_time_lag5', 'avg_recovery_time_lag6', 'avg_recovery_time_lag7', 'near_collision_15m_lag4',
    'near_collision_15m_lag5', 'near_collision_15m_lag6', 'near_collision_15m_lag7', 'charge_queue_length_roll7_std',
    'charge_queue_length_roll10_mean', 'low_battery_ratio_lag4', 'low_battery_ratio_lag5', 'low_battery_ratio_lag7',
    'blocked_path_15m_lag4', 'blocked_path_15m_lag5', 'blocked_path_15m_lag6', 'blocked_path_15m_lag7',
    'label_print_queue_delta_start', 'robot_charging_lag15', 'low_battery_ratio_lag12', 'low_battery_ratio_lag15',
    'charge_queue_length_lag12', 'charge_queue_length_lag15', 'avg_charge_wait_lag12', 'avg_charge_wait_lag15',
    'congestion_score_lag12', 'max_zone_density_lag15', 'blocked_path_15m_lag12', 'fault_count_15m_lag4',
    'fault_count_15m_lag5', 'fault_count_15m_lag6', 'fault_count_15m_lag7', 'charge_queue_length_lag4',
    'charge_queue_length_lag5', 'charge_queue_length_lag6', 'charge_queue_length_lag7', 'charge_queue_length_roll7_mean',
    'blocked_path_15m_lag15', 'near_collision_15m_lag12', 'near_collision_15m_lag15', 'fault_count_15m_lag12',
    'fault_count_15m_lag15', 'avg_recovery_time_lag12', 'avg_recovery_time_lag15', 'task_reassign_15m_lag12',
    'task_reassign_15m_lag15', 'replenishment_overlap_lag12', 'replenishment_overlap_lag15', 'charge_queue_length_position_in_range',
    'avg_charge_wait_position_in_range', 'congestion_score_position_in_range', 'blocked_path_15m_position_in_range', 'near_collision_15m_position_in_range',
    'fault_count_15m_position_in_range', 'avg_recovery_time_position_in_range', 'task_reassign_15m_position_in_range', 'replenishment_overlap_position_in_range',
    'label_print_queue_position_in_range', 'replenishment_overlap_vs_cummin', 'staging_area_util_vs_cummax', 'battery_mean_position_in_range',
    'low_battery_ratio_position_in_range', 'label_print_queue_lag15', 'charge_queue_length_roll20_std', 'charge_queue_length_vs_start',
    'charge_queue_length_delta_start', 'avg_charge_wait_vs_start', 'avg_charge_wait_delta_start'
]
features_pruned = [f for f in features_base if f not in zero_imp_features]
print(f"Features reduced from {len(features_base)} to {len(features_pruned)} (-{len(features_base)-len(features_pruned)})")

In [ ]:
# Cell 7: cat_params_base 설정 (Exp 66 그대로)
cat_params_base = {
    'iterations': 1441, 'learning_rate': 0.024382726628741795, 'depth': 7,
    'l2_leaf_reg': 4.329713228202991, 'bagging_temperature': 0.15517607913494932,
    'loss_function': 'MAE', 'eval_metric': 'MAE', 'verbose': False,
    'task_type': 'GPU', 'devices': '0',
    'early_stopping_rounds': 100
}

In [ ]:
# Cell 8: TRANSFORMS 딕셔너리 정의 + gkf 초기화
TRANSFORMS = {
    'log1p': {
        'forward': lambda y: np.log1p(y),
        'inverse': lambda p: np.expm1(p)
    },
    'sqrt': {
        'forward': lambda y: np.sqrt(y),
        'inverse': lambda p: np.maximum(p, 0)**2
    },
    'none': {
        'forward': lambda y: y,
        'inverse': lambda p: p
    }
}

gkf = GroupKFold(n_splits=5)

In [ ]:
# Cell 9: [Stage 1] 타겟 변환 3가지 비교 (seed=42 단독, 빠른 탐색)
transform_results = {}

for t_name, t_func in TRANSFORMS.items():
    print(f"\n{'='*50}")
    print(f"[Stage 1] target_transform={t_name} 탐색 중...")

    lgb_params = {
        'learning_rate': 0.03, 'n_estimators': 1500,
        'max_depth': 7, 'num_leaves': 127, # Exp 66 최적값
        'verbose': -1, 'objective': 'huber',
        'subsample': 0.75, 'colsample_bytree': 0.65,
        'subsample_freq': 1, 'random_state': 42
    }
    xgb_params = {
        'learning_rate': 0.03, 'n_estimators': 1000,
        'max_depth': 7, 'objective': 'reg:absoluteerror',
        'verbosity': 0, 'early_stopping_rounds': 100,
        'subsample': 0.75, 'colsample_bytree': 0.65,
        'random_state': 42
    }
    cat_params = {**cat_params_base, 'random_seed': 42}

    oof_lgb_s = np.zeros(len(train))
    oof_xgb_s = np.zeros(len(train))
    oof_cat_s = np.zeros(len(train))

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(train, train[TARGET], groups=train['layout_id'])):
        X_tr, y_tr = train.iloc[tr_idx].copy(), train.iloc[tr_idx][TARGET]
        X_val, y_val = train.iloc[val_idx].copy(), train.iloc[val_idx][TARGET]

        X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
        X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True)
        feats = features_pruned + te_cols

        # 변환 적용
        y_tr_t = t_func['forward'](y_tr)
        y_val_t = t_func['forward'](y_val)

        m_lgb = LGBMRegressor(**lgb_params).fit(
            X_tr[feats], y_tr_t,
            eval_set=[(X_val[feats], y_val_t)],
            callbacks=[lgb.early_stopping(100)]
        )
        oof_lgb_s[val_idx] = t_func['inverse'](m_lgb.predict(X_val[feats]))

        m_xgb = XGBRegressor(**xgb_params).fit(
            X_tr[feats], y_tr_t,
            eval_set=[(X_val[feats], y_val_t)],
            verbose=False
        )
        oof_xgb_s[val_idx] = t_func['inverse'](m_xgb.predict(X_val[feats]))

        m_cat = CatBoostRegressor(**cat_params).fit(
            X_tr[feats], y_tr_t,
            eval_set=[(X_val[feats], y_val_t)]
        )
        oof_cat_s[val_idx] = t_func['inverse'](m_cat.predict(X_val[feats]))

        print(f"Fold {fold+1} Done", end=' | ', flush=True)

    # 음수 클리핑
    oof_lgb_s = np.maximum(oof_lgb_s, 0)
    oof_xgb_s = np.maximum(oof_xgb_s, 0)
    oof_cat_s = np.maximum(oof_cat_s, 0)

    mae_lgb_s = mean_absolute_error(train[TARGET], oof_lgb_s)
    mae_xgb_s = mean_absolute_error(train[TARGET], oof_xgb_s)
    mae_cat_s = mean_absolute_error(train[TARGET], oof_cat_s)
    inv = np.array([1/mae_lgb_s, 1/mae_xgb_s, 1/mae_cat_s])
    w = inv / inv.sum()
    oof_ens_s = oof_lgb_s*w[0] + oof_xgb_s*w[1] + oof_cat_s*w[2]
    oof_mae_s = mean_absolute_error(train[TARGET], np.maximum(oof_ens_s, 0))

    transform_results[t_name] = oof_mae_s
    print(f"\n  transform={t_name} → OOF MAE: {oof_mae_s:.4f} "
          f"(LGB:{mae_lgb_s:.4f}, XGB:{mae_xgb_s:.4f}, CAT:{mae_cat_s:.4f})")
    gc.collect()

In [ ]:
# Cell 10: Stage 1 결과 요약 + best_transform 확정
best_transform = min(transform_results, key=transform_results.get)

print(f"\n{'='*55}")
print(f"[Stage 1 결과 요약]")
for t_name, mae in sorted(transform_results.items(), key=lambda x: x[1]):
    marker = " ← 최적" if t_name == best_transform else ""
    print(f"  transform={t_name:8s}  OOF MAE: {mae:.4f}{marker}")
print(f"\n최적 변환: {best_transform}")
print(f"{'='*55}\n")

In [ ]:
# Cell 11: [Stage 2] best_transform으로 10-Seed 풀 앙상블
print(f"[Stage 2] best_transform={best_transform}으로 10-Seed 풀 앙상블 시작...")

t_func = TRANSFORMS[best_transform]
seeds = [42, 123, 2026, 777, 1004, 314, 555, 888, 999, 1337]
oof_seed_ensembles = np.zeros(len(train))
test_preds_total = np.zeros(len(test))

for seed_idx, seed in enumerate(seeds):
    print(f"\n{'='*30} Seed {seed} ({seed_idx+1}/10) {'='*30}")

    oof_lgb = np.zeros(len(train))
    oof_xgb = np.zeros(len(train))
    oof_cat = np.zeros(len(train))

    seed_lgb_preds = np.zeros(len(test))
    seed_xgb_preds = np.zeros(len(test))
    seed_cat_preds = np.zeros(len(test))

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(train, train[TARGET], groups=train['layout_id'])):
        X_tr, y_tr = train.iloc[tr_idx].copy(), train.iloc[tr_idx][TARGET]
        X_val, y_val = train.iloc[val_idx].copy(), train.iloc[val_idx][TARGET]

        X_tr, X_val, te_cols = apply_smoothed_te(X_tr, X_val, TARGET, k=30)
        X_tr.fillna(0, inplace=True); X_val.fillna(0, inplace=True)
        feats = features_pruned + te_cols

        _, X_test_fold, _ = apply_smoothed_te(X_tr, test.copy(), TARGET, k=30)
        X_test_fold.fillna(0, inplace=True)

        # 변환 적용
        y_tr_t = t_func['forward'](y_tr)
        y_val_t = t_func['forward'](y_val)

        # LGBM
        lgb_params = {
            'learning_rate': 0.03, 'n_estimators': 1500, 'max_depth': 7, 'num_leaves': 127,
            'verbose': -1, 'objective': 'huber', 'subsample': 0.75, 'colsample_bytree': 0.65,
            'subsample_freq': 1, 'random_state': seed
        }
        m_lgb = LGBMRegressor(**lgb_params).fit(
            X_tr[feats], y_tr_t,
            eval_set=[(X_val[feats], y_val_t)],
            callbacks=[lgb.early_stopping(100)]
        )
        oof_lgb[val_idx] = t_func['inverse'](m_lgb.predict(X_val[feats]))
        seed_lgb_preds += t_func['inverse'](m_lgb.predict(X_test_fold[feats])) / 5

        # XGB
        xgb_params = {
            'learning_rate': 0.03, 'n_estimators': 1000, 'max_depth': 7,
            'objective': 'reg:absoluteerror', 'verbosity': 0, 'early_stopping_rounds': 100,
            'subsample': 0.75, 'colsample_bytree': 0.65, 'random_state': seed
        }
        m_xgb = XGBRegressor(**xgb_params).fit(
            X_tr[feats], y_tr_t,
            eval_set=[(X_val[feats], y_val_t)],
            verbose=False
        )
        oof_xgb[val_idx] = t_func['inverse'](m_xgb.predict(X_val[feats]))
        seed_xgb_preds += t_func['inverse'](m_xgb.predict(X_test_fold[feats])) / 5

        # CatBoost
        cat_params = {**cat_params_base, 'random_seed': seed}
        m_cat = CatBoostRegressor(**cat_params).fit(
            X_tr[feats], y_tr_t,
            eval_set=[(X_val[feats], y_val_t)]
        )
        oof_cat[val_idx] = t_func['inverse'](m_cat.predict(X_val[feats]))
        seed_cat_preds += t_func['inverse'](m_cat.predict(X_test_fold[feats])) / 5

        print(f"Fold {fold+1} Done", end=' | ', flush=True)

    oof_lgb = np.maximum(oof_lgb, 0)
    oof_xgb = np.maximum(oof_xgb, 0)
    oof_cat = np.maximum(oof_cat, 0)

    mae_lgb = mean_absolute_error(train[TARGET], oof_lgb)
    mae_xgb = mean_absolute_error(train[TARGET], oof_xgb)
    mae_cat = mean_absolute_error(train[TARGET], oof_cat)

    inv_mae_sum = (1/mae_lgb) + (1/mae_xgb) + (1/mae_cat)
    w_lgb, w_xgb, w_cat = (1/mae_lgb)/inv_mae_sum, (1/mae_xgb)/inv_mae_sum, (1/mae_cat)/inv_mae_sum

    seed_oof = (oof_lgb * w_lgb) + (oof_xgb * w_xgb) + (oof_cat * w_cat)
    oof_seed_ensembles += seed_oof / 10

    seed_test_preds = (seed_lgb_preds * w_lgb) + (seed_xgb_preds * w_xgb) + (seed_cat_preds * w_cat)
    test_preds_total += seed_test_preds / 10

    print(f"\nSeed {seed} MAE: {mean_absolute_error(train[TARGET], seed_oof):.4f} (LGB:{mae_lgb:.4f}, XGB:{mae_xgb:.4f}, CAT:{mae_cat:.4f})", flush=True)
    gc.collect()

In [ ]:
# Cell 12: Final Model 학습 → submission_67.csv 저장
test_preds_total = np.maximum(test_preds_total, 0)
pd.DataFrame({'ID': test['ID'], TARGET: test_preds_total}).to_csv(OUTPUT_PATH, index=False)
print(f"\nExperiment 67 complete. Submission saved to {OUTPUT_PATH}")

In [ ]:
# Cell 13: 최종 결과 비교 출력
total_mae = mean_absolute_error(train[TARGET], oof_seed_ensembles)
seen_mae = mean_absolute_error(train[train['is_seen']][TARGET], oof_seed_ensembles[train['is_seen']])
unseen_mae = mean_absolute_error(train[~train['is_seen']][TARGET], oof_seed_ensembles[~train['is_seen']])

print("\n" + "=" * 65)
print(f"  [Exp 66 Baseline]   OOF MAE : 8.7250  |  LB : 10.156")
print(f"  [Stage 1 비교]")
for t_name, mae in sorted(transform_results.items(), key=lambda x: x[1]):
    marker = " ← 채택" if t_name == best_transform else ""
    print(f"    transform={t_name:8s}  seed=42 OOF: {mae:.4f}{marker}")
print(f"  [Exp 67 최종]       OOF MAE : {total_mae:.4f}  "
      f"(Δ{total_mae - 8.7250:+.4f})")
print(f"  [채택된 변환]       {best_transform}")
print(f"  [Seen  MAE]        : {seen_mae:.4f}")
print(f"  [Unseen MAE]       : {unseen_mae:.4f}")
print(f"  [제출 파일]        : submission_67.csv → Google Drive 저장 완료")
print("=" * 65)